# 일곱 모델 한자리에서 비교

앞의 일곱 페이지에서 하나씩 본 모델을 **같은 데이터, 같은 자리**로 나란히 놓는다.

결과 이미지는 미리 뽑아 저장소에 올려둔 것을 불러온다. **GPU 가 필요 없다.**

| # | 모델 | 연도 | 구조 | 손실 |
|---|---|---|---|---|
| 1 | SRCNN | 2014 | conv 3장 | MSE |
| 2 | VDSR | 2016 | conv 20장 + 잔차 | MSE |
| 3 | EDSR | 2017 | CNN (residual block) | L1 |
| 4 | SRGAN | 2017 | CNN + GAN | MSE + VGG + 적대적 |
| 5 | ESRGAN | 2018 | CNN (RRDB) + GAN | L1 + VGG + RaGAN |
| 6 | SwinIR | 2021 | Transformer (Swin) | L1 |
| 7 | HAT | 2023 | Transformer + 채널·중첩 어텐션 | L1 |

## 1. 준비

In [ ]:
import sys, json, urllib.request

LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
urllib.request.urlretrieve(f'{LIB}/sr_utils.py', 'sr_utils.py')
sys.modules.pop('sr_utils', None)
from sr_utils import *

from matplotlib.patches import Rectangle

RES = f'{BASE}/results/comparison'
V = 'v3'          # 내려받은 파일 캐시 폴더. 결과가 바뀌면 이 값을 올린다
META = json.load(open(fetch(f'{RES}/meta.json', f'{V}/meta.json')))
ORDER = ['LR', 'Bicubic', 'SRCNN', 'VDSR', 'EDSR',
         'SRGAN', 'ESRGAN', 'SwinIR', 'HAT', 'HR']


def load(scene, name):
    return imageio.imread(fetch(f'{RES}/{scene}/{name}.png', f'{V}/{scene}/{name}.png'))


def compare_scene(scene, center=None, size=None):
    """center=(cx, cy) 로 확대 위치를, size 로 창 크기를 정한다.

    좌표는 **결과 그림(SR, LR x3) 기준**이다. validation 은 384px, 인천은 1800px.
    안 주면 아래 CENTER / SIZE 의 기본값을 쓴다.
    """
    m = META[scene]
    cx, cy = center or CENTER[scene]
    size = size or SIZE[scene]
    ox, oy = m['origin']; ew, eh = m['extent']

    # 저장해 둔 영역 안으로 당긴다 (인천은 SR 1800 중 가운데 768 만 올려 뒀다)
    x = int(cx - ox - size // 2); y = int(cy - oy - size // 2)
    x = max(0, min(x, ew - size)); y = max(0, min(y, eh - size))
    ax_, ay_ = x + ox, y + oy                      # 실제로 보게 된 SR 좌표
    if (ax_ + size // 2, ay_ + size // 2) != (cx, cy):
        print(f'  {scene}: 저장 영역(SR {ox}~{ox+ew}) 밖이라 '
              f'center=({ax_ + size//2}, {ay_ + size//2}) 로 당겼다')

    names = [n for n in ORDER if n != 'HR' or m['has_hr']]
    ov = load(scene, 'overview')
    r = m['overview'][0] / m['full'][0]             # 전체 그림을 줄인 비율

    fig = plt.figure(figsize=(2.05 * len(names), 6.6))
    gs = fig.add_gridspec(2, len(names), height_ratios=[1.9, 1])
    a = fig.add_subplot(gs[0, :])
    a.imshow(ov)
    a.add_patch(Rectangle((ax_ * r, ay_ * r), size * r, size * r,
                          fill=False, ec='#ffcc00', lw=2))
    a.add_patch(Rectangle((ox * r, oy * r), ew * r, eh * r,
                          fill=False, ec='#8888ff', lw=1, ls='--'))
    a.set_xticks([]); a.set_yticks([])
    a.set_title(f"{m['title']}   (yellow = zoom, blue dashed = stored area, "
                f"center=({ax_ + size//2}, {ay_ + size//2}), size={size})", fontsize=10)

    for j, n in enumerate(names):
        b = fig.add_subplot(gs[1, j])
        b.imshow(load(scene, n)[y:y + size, x:x + size], interpolation='nearest')
        b.set_title(n, fontsize=9); b.set_xticks([]); b.set_yticks([])
    plt.tight_layout(); plt.show()


MODELS = [n for n in ORDER if n not in ('LR', 'Bicubic', 'HR')]
ORDER_M = ['Bicubic'] + MODELS
COLORS = ['#8c6bb1', '#4f9f6f', '#2f6f9f', '#d98c3f', '#c0504d', '#3f9fa8', '#7f5f3f']


def show_method(scene, name, center=None, size=None):
    """한 방법만 골라 4x2 로 본다 — 각 모델 페이지의 show_results 와 같은 모양.

    윗줄 = 전체(노란 네모가 확대 자리), 아랫줄 = 그 자리 확대.
    열은 Original LR / Bicubic / 해당 방법 / Target HR 이다 (인천은 정답이 없어 3열).
    center, size 는 compare_scene 과 같은 규칙이다 (SR 좌표).
    """
    m = META[scene]
    cx, cy = center or CENTER[scene]
    size = size or SIZE[scene]
    ox, oy = m['origin']; ew, eh = m['extent']
    x = max(0, min(int(cx - ox - size // 2), ew - size))
    y = max(0, min(int(cy - oy - size // 2), eh - size))

    panels = [('Original LR', load(scene, 'LR')), ('Bicubic', load(scene, 'Bicubic')),
              (name, load(scene, name))]
    if m['has_hr']:
        panels.append(('Target HR', load(scene, 'HR')))

    fig, ax = plt.subplots(2, len(panels), figsize=(2.9 * len(panels), 6.4),
                           squeeze=False)
    for j, (n, im) in enumerate(panels):
        a = ax[0][j]
        a.imshow(im, interpolation='nearest')
        a.add_patch(Rectangle((x, y), size, size, fill=False, ec='#ffcc00', lw=1.6))
        a.set_title(n, fontsize=9)
        a.set_xticks([]); a.set_yticks([])

        b = ax[1][j]
        b.imshow(im[y:y + size, x:x + size], interpolation='nearest')
        b.set_xticks([]); b.set_yticks([])

    ax[0][0].set_ylabel('full', fontsize=10)
    ax[1][0].set_ylabel(f'zoom {size}px\ncenter=({x + ox + size // 2}, '
                        f'{y + oy + size // 2})', fontsize=9)
    fig.suptitle(f"{m['title']}  —  {name}", fontsize=10)
    plt.tight_layout(); plt.show()


# ── 여기를 바꾸면 확대 위치가 바뀐다 (SR 좌표) ─────────────────────────
CENTER = {'val1': (137, 236), 'val2': (313, 346),
          'test1': (800, 800), 'test2': (800, 800)}
SIZE   = {'val1': 55, 'val2': 55, 'test1': 70, 'test2': 70}

print('준비 완료 —', ', '.join(META))
for k, m in META.items():
    ox, oy = m['origin']; ew, eh = m['extent']
    print(f"  {k:6s} SR {m['full'][1]}x{m['full'][0]}   "
          f"조절 가능 범위 x,y = {ox}~{ox + ew}")

## 2. 결과 — validation

정답(HR)이 있는 패치다. 맨 왼쪽이 입력, 맨 오른쪽이 정답이다.

확대 위치를 바꾸려면 위 셀의 `CENTER` / `SIZE` 를 고치거나,
`compare_scene('val1', center=(200, 250), size=80)` 처럼 직접 넘기면 된다.

In [ ]:
for s in ('val1', 'val2'):
    compare_scene(s)

### 2-1. 방법별로 하나씩

위 그림은 확대한 자리만 열 개를 늘어놓은 것이라 전체 맥락이 안 보인다.
여기서는 각 모델 페이지의 `show_results` 와 **똑같은 4x2** 로 한 방법씩 본다.

열은 `Original LR / Bicubic / 해당 방법 / Target HR`, 윗줄이 전체, 아랫줄이 확대다.
한 셀이 모델 일곱 개 그림을 낸다. 하나만 보려면 `show_method('val1', 'EDSR')`.

In [ ]:
for n in MODELS:
    show_method('val1', n)


In [ ]:
for n in MODELS:
    show_method('val2', n)


## 3. 최종 테스트 — 인천

정답이 없는 실제 Sentinel-2 촬영본이다. 점수는 못 내고 눈으로만 비교한다.

방법별 4x2 도 같은 함수로 본다: `show_method('test1', 'HAT')`. 정답이 없어 열은 세 개다.


In [ ]:
for s in ('test1', 'test2'):
    compare_scene(s)

In [ ]:
for n in MODELS:
    show_method('test1', n)


## 4. 정량 비교

검증 10패치 전체 평균이다. 입력은 모두 같다.

In [ ]:
import pandas as pd

df = pd.read_csv(fetch(f'{RES}/metrics.csv', f'{V}/metrics.csv')).set_index('model')
print(df.to_string(float_format=lambda v: f'{v:.4f}'))

In [ ]:
base = df.loc['Bicubic']
m = df.drop('Bicubic')
idx = np.arange(len(m))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
for a, col, ttl in [(ax[0], 'PSNR', 'PSNR (dB)'), (ax[1], 'SSIM', 'SSIM')]:
    a.bar(idx, m[col], 0.6, color='#4f7fa8')
    a.axhline(base[col], color='#888', ls='--', lw=1.4, label='Bicubic')
    a.set_xticks(idx); a.set_xticklabels(m.index, rotation=30, fontsize=9)
    a.set_title(ttl); a.grid(axis='y', alpha=.3); a.legend(fontsize=8)
    lo, hi = min(m[col].min(), base[col]), max(m[col].max(), base[col])
    a.set_ylim(lo - (hi - lo) * .3, hi + (hi - lo) * .2)

ax[2].scatter(m.params_M, m.PSNR, s=70, color='#c96a5b')
for n, r in m.iterrows():
    ax[2].annotate(n, (r.params_M, r.PSNR), fontsize=8,
                   xytext=(4, 4), textcoords='offset points')
ax[2].axhline(base.PSNR, color='#888', ls='--', lw=1.4)
ax[2].set_xscale('log'); ax[2].set_xlabel('parameters (M, log)')
ax[2].set_ylabel('PSNR (dB)'); ax[2].set_title('PSNR vs model size'); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

### 4-1. 패치별로 뜯어보기

위 막대는 10패치 평균이다. 평균만 보면 모델 사이 차이가 0.7 dB 안쪽이라 작아 보이는데,
**패치 하나하나로 내려가면 그림이 다르다.** 왼쪽 두 그림은 검증 10패치 각각의 점수이고,
오른쪽은 bicubic 대비 이득을 모델별로 모아 놓은 것이다.

패치 사이 편차(세로 방향)가 모델 사이 차이(선 간격)보다 훨씬 크다 —
어떤 장면이냐가 어떤 모델이냐보다 점수를 더 많이 좌우한다.

In [ ]:
per = pd.read_csv(fetch(f'{RES}/metrics_per_patch.csv', f'{V}/metrics_per_patch.csv'))
piv = {c: per.pivot(index='patch', columns='model', values=c)[ORDER_M] for c in ('PSNR', 'SSIM')}
# 같은 씬에서 온 패치가 둘이면 이름이 겹친다. -1, -2 를 붙여 구분한다
names = [s.replace('AOI_', '').rsplit('_y', 1)[0] for s in piv['PSNR'].index]
seen, patches = {}, []
for n in names:
    seen[n] = seen.get(n, 0) + 1
    patches.append(n if names.count(n) == 1 else f'{n}-{seen[n]}')
x = np.arange(len(patches))

fig, ax = plt.subplots(1, 3, figsize=(19, 4.6))
for a, col in zip(ax[:2], ('PSNR', 'SSIM')):
    d = piv[col]
    a.plot(x, d['Bicubic'], color='#888', ls='--', lw=2, marker='o', ms=4, label='Bicubic')
    for i, m in enumerate(MODELS):
        a.plot(x, d[m], color=COLORS[i], lw=1.4, marker='o', ms=4, label=m)
    a.set_xticks(x); a.set_xticklabels(patches, rotation=45, ha='right', fontsize=8)
    a.set_title(f'{col} per patch'); a.grid(alpha=.3)
ax[0].set_ylabel('PSNR (dB)'); ax[1].set_ylabel('SSIM')
ax[0].legend(fontsize=8, ncol=2)

# 세 번째: bicubic 대비 이득. 패치마다 얼마나 흔들리는지 본다 (그림 글자는 영문)
d = piv['PSNR'].sub(piv['PSNR']['Bicubic'], axis=0)[MODELS]
ax[2].boxplot([d[m] for m in MODELS], labels=MODELS, widths=.6,
              medianprops=dict(color='#c96a5b', lw=2))
for i, m in enumerate(MODELS):
    ax[2].scatter(np.full(len(d), i + 1) + np.random.uniform(-.12, .12, len(d)),
                  d[m], s=14, color=COLORS[i], alpha=.7, zorder=3)
ax[2].axhline(0, color='#888', ls='--', lw=1.4)
ax[2].set_xticklabels(MODELS, rotation=30, fontsize=9)
ax[2].set_ylabel('ΔPSNR vs Bicubic (dB)')
ax[2].set_title('gain over Bicubic, per patch (n=10)'); ax[2].grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

print(piv['PSNR'].round(2).to_string())


## 5. 정리

**모델을 키운 만큼 오르지 않는다.** 가장 작은 SRCNN(0.06M)과 가장 큰 HAT(20.81M)은
파라미터가 363배 차이인데 PSNR 차이는 0.66 dB 다. 오른쪽 그래프가 가로축 로그인데도
거의 평평하다.

**GAN 계열은 지표가 낮다.** SRGAN·ESRGAN 은 화소 정확도 대신 그럴듯한 질감을 만들도록
학습된다. 위 확대 그림에서 두 모델만 눈에 띄게 선명하고, 대신 PSNR 은 최하위다.
ESRGAN 은 bicubic 보다도 낮다. 어느 쪽이 "좋은" 결과인지는 용도가 정한다.

**병목은 모델이 아니다.** 학습과 검증은 합성 저해상도(g_LR)를 쓰지만 인천은 실제
Sentinel-2 촬영본이다. 실제 관측과 합성 사이의 차이가 남아 있는 한 구조를 키우는
것만으로는 넘기 어렵다.